# Homogeneización de bases de datos de celulosa (δ13C, δ15N, %C, %N)

Este notebook:

1. Carga los 4 archivos (`Sumaco_cellulose.xlsx`, `Galeras_cellulose.xlsx`, `SV_cellulose.xlsx`, `iWUE_OYC_GUA.xlsx`).
2. Homogeneiza nombres de columnas a un esquema común.
3. Añade la columna `Site` a todas las filas.
4. Añade `Elevation_m` usando `elevation_dict.py` (editable).
5. **Verifica y corrige unidades mezcladas**: varias columnas de `[C]%`, `[C_b]%` y `[N]` tienen algunos valores en fracción (0–1) y otros ya en porcentaje (0–100). Se detectan y corrigen automáticamente.
6. Calcula cuántos árboles (identificados por `new_TreeID`) tienen mediciones **tanto en 2006 como en 2025**, por sitio.

> **Antes de correr**: coloca los 5 archivos (4 `.xlsx` + `elevation_dict.py`) en la misma carpeta que este notebook, o ajusta `CARPETA_DATOS` abajo.


In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Carpeta donde están los 4 xlsx y elevation_dict.py (ajusta si es necesario)
CARPETA_DATOS = Path(".")
sys.path.insert(0, str(CARPETA_DATOS))

from elevation_dict import elevation_dict

print(f"{len(elevation_dict)} combinaciones (Site, Plot) definidas en elevation_dict.py")


34 combinaciones (Site, Plot) definidas en elevation_dict.py


## 1. Cargar los 4 archivos originales

In [2]:
df_sumaco  = pd.read_excel(CARPETA_DATOS / "Sumaco_cellulose.xlsx")
df_galeras = pd.read_excel(CARPETA_DATOS / "Galeras_cellulose.xlsx")
df_sv      = pd.read_excel(CARPETA_DATOS / "SV_cellulose.xlsx")
df_iwue    = pd.read_excel(CARPETA_DATOS / "iWUE_OYC_GUA.xlsx")

for nombre, d in [("Sumaco", df_sumaco), ("Galeras", df_galeras),
                  ("SV", df_sv), ("iWUE_OYC_GUA", df_iwue)]:
    print(f"{nombre:15s} -> {d.shape[0]:4d} filas, {d.shape[1]:2d} columnas")


Sumaco          ->   91 filas, 13 columnas
Galeras         ->  206 filas, 19 columnas
SV              ->  223 filas, 20 columnas
iWUE_OYC_GUA    ->  152 filas, 24 columnas


## 2. Verificación de unidades: ¡hay valores mezclados en fracción y en %!

Ejemplo concreto en `Galeras_cellulose.xlsx`: la misma muestra `p23-557` aparece
duplicada, una vez con `[C]% = 0.4555` (fracción) y otra con `[C]% = 43.36` (porcentaje real).
Esto ocurre en varias columnas (`[C]%`, `[C_b]%`, `[N]`) y en varios archivos.
Abajo se muestra el caso y luego se corrige de forma sistemática: cualquier valor
**menor a 1** en estas columnas se interpreta como fracción y se multiplica por 100.

In [3]:
# Evidencia del problema de unidades mezcladas
ejemplo = df_galeras[df_galeras["Sample-ID"] == "p23-557"][
    ["Sample-ID", "[C]%", "[C_b]%", "[N]"]
]
print("Ejemplo de unidades mezcladas para la misma muestra:")
print(ejemplo.to_string(index=False))


Ejemplo de unidades mezcladas para la misma muestra:
Sample-ID      [C]%   [C_b]%      [N]
  p23-557  0.455530 0.485531 0.019124
  p23-557 43.360479 0.485531 0.019124


## 3. Funciones de homogeneización

In [4]:
def normaliza_plot(v):
    """Convierte '09', 'p09', 9, 9.0 -> 9 (int). Devuelve NaN si no se puede."""
    if pd.isna(v):
        return np.nan
    if isinstance(v, str):
        v = v.strip().lower().lstrip("p")
        if v in ("", "nan"):
            return np.nan
        try:
            return int(float(v))
        except ValueError:
            return np.nan
    try:
        return int(v)
    except (ValueError, TypeError):
        return np.nan


def homogeniza_porcentaje(serie, nombre_col, umbral=1.0):
    """
    [C]%, [C_b]% y [N] vienen mezcladas: algunas filas en fracción (0-1),
    otras ya en % (0-100). Si el valor es < umbral, se asume fracción
    y se multiplica por 100. Imprime cuántos valores se corrigieron.
    """
    s = serie.copy()
    mask = s.notna() & (s.abs() < umbral)
    n_corr = int(mask.sum())
    s.loc[mask] = s.loc[mask] * 100
    print(f"  -> {nombre_col}: {n_corr} valores estaban en fracción y se multiplicaron x100")
    return s


# Esquema final común para las 4 bases
COLUMNAS_FINALES = [
    "Site", "Plot", "PlotID", "Year", "Sample_ID", "old_treeID", "new_TreeID",
    "family", "genus", "species", "Type",
    "C_percent", "d13C_permil", "N_percent", "Cb_percent", "d15N_permil",
    "bulk_d13C_permil", "herbarium_id", "source_file",
]


def homogeniza(df, site_fijo=None, col_site=None, plot_col="Plot",
               sample_col="Sample-ID", source=""):
    d = pd.DataFrame(index=df.index)

    d["Site"] = site_fijo if site_fijo is not None else df[col_site]
    d["Plot"] = df[plot_col].apply(normaliza_plot)
    d["PlotID"] = d["Site"].astype(str) + "_" + d["Plot"].astype("Int64").astype(str)
    d["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")

    d["Sample_ID"] = df[sample_col] if sample_col in df.columns else np.nan
    d["old_treeID"] = pd.to_numeric(df.get("old_treeID"), errors="coerce")
    d["new_TreeID"] = pd.to_numeric(df.get("new_TreeID"), errors="coerce")

    d["family"] = df.get("family")
    d["genus"] = df.get("genus")
    d["species"] = df.get("species") if "species" in df.columns else df.get("species2")
    d["Type"] = df.get("Type")

    d["C_percent"] = (homogeniza_porcentaje(df["[C]%"], f"{source}:[C]%")
                       if "[C]%" in df.columns else np.nan)
    d["d13C_permil"] = df.get("δ13C (‰ v.s.V-PDB)")

    d["N_percent"] = (homogeniza_porcentaje(df["[N]"], f"{source}:[N]")
                       if "[N]" in df.columns else np.nan)
    d["Cb_percent"] = (homogeniza_porcentaje(df["[C_b]%"], f"{source}:[C_b]%")
                        if "[C_b]%" in df.columns else np.nan)

    d["d15N_permil"] = df.get("δ15N (‰ v.s. V-PDB)")
    d["bulk_d13C_permil"] = df.get("bulk_δ13C (‰ v.s.V-PDB)")
    d["herbarium_id"] = (df.get("JH herbarium collections")
                          if "JH herbarium collections" in df.columns
                          else df.get("herbarium specimen"))
    d["source_file"] = source
    return d[COLUMNAS_FINALES]


## 4. Aplicar la homogeneización a cada base y concatenar

- `Sumaco_cellulose.xlsx` y `Galeras_cellulose.xlsx` -> sitio fijo (`Sumaco`, `Galeras`).
- `SV_cellulose.xlsx` -> sitio fijo `SV` (sus parcelas 68, 69, 70, 72, 74 **no existen todavía**
  en `elevation_dict.py`; si `SV` corresponde a un sitio con otro nombre, cámbialo abajo).
- `iWUE_OYC_GUA.xlsx` -> ya trae su propia columna `Site` (`Oyacachi` / `Guacamayos`), se respeta tal cual.

In [5]:
print("Homogeneizando Sumaco...")
h_sumaco = homogeniza(df_sumaco, site_fijo="Sumaco", source="Sumaco_cellulose.xlsx")

print("\nHomogeneizando Galeras...")
h_galeras = homogeniza(df_galeras, site_fijo="Galeras", source="Galeras_cellulose.xlsx")

print("\nHomogeneizando SV...")
h_sv = homogeniza(df_sv, site_fijo="SV", source="SV_cellulose.xlsx")

print("\nHomogeneizando iWUE (Oyacachi/Guacamayos)...")
h_iwue = homogeniza(df_iwue, col_site="Site", source="iWUE_OYC_GUA.xlsx")

df_all = pd.concat([h_sumaco, h_galeras, h_sv, h_iwue], ignore_index=True)
print(f"\nTotal filas combinadas: {len(df_all)}")
print(df_all["Site"].value_counts(dropna=False))


Homogeneizando Sumaco...
  -> Sumaco_cellulose.xlsx:[C]%: 5 valores estaban en fracción y se multiplicaron x100

Homogeneizando Galeras...
  -> Galeras_cellulose.xlsx:[C]%: 52 valores estaban en fracción y se multiplicaron x100
  -> Galeras_cellulose.xlsx:[N]: 82 valores estaban en fracción y se multiplicaron x100
  -> Galeras_cellulose.xlsx:[C_b]%: 82 valores estaban en fracción y se multiplicaron x100

Homogeneizando SV...
  -> SV_cellulose.xlsx:[C]%: 57 valores estaban en fracción y se multiplicaron x100
  -> SV_cellulose.xlsx:[N]: 48 valores estaban en fracción y se multiplicaron x100
  -> SV_cellulose.xlsx:[C_b]%: 48 valores estaban en fracción y se multiplicaron x100

Homogeneizando iWUE (Oyacachi/Guacamayos)...
  -> iWUE_OYC_GUA.xlsx:[C]%: 0 valores estaban en fracción y se multiplicaron x100
  -> iWUE_OYC_GUA.xlsx:[N]: 41 valores estaban en fracción y se multiplicaron x100
  -> iWUE_OYC_GUA.xlsx:[C_b]%: 41 valores estaban en fracción y se multiplicaron x100

Total filas combina

## 5. Verificación posterior de unidades

Tras la corrección, `C_percent` y `Cb_percent` deberían moverse en un rango
razonable para tejido vegetal (~20–70 %) y `N_percent` en (~0.3–5 %).
Revisamos mín/máx para confirmar que ya no queden fracciones sueltas.

In [6]:
resumen_unidades = df_all[["C_percent", "Cb_percent", "N_percent"]].agg(["min", "max", "mean", "count"]).T
print(resumen_unidades)

# Alerta si sigue quedando algún valor sospechosamente bajo (<1) sin corregir
for col in ["C_percent", "Cb_percent", "N_percent"]:
    sospechosos = df_all[(df_all[col].notna()) & (df_all[col].abs() < 1)]
    if len(sospechosos):
        print(f"⚠ {col}: aún quedan {len(sospechosos)} valores < 1 -> revisar manualmente")
    else:
        print(f"✓ {col}: sin valores fuera de rango tras la corrección")


                  min        max       mean  count
C_percent   19.519720  76.936851  44.129171  657.0
Cb_percent  -3.431583  53.852797  43.595403  267.0
N_percent    1.025192   4.459619   2.206396  267.0
✓ C_percent: sin valores fuera de rango tras la corrección
✓ Cb_percent: sin valores fuera de rango tras la corrección
✓ N_percent: sin valores fuera de rango tras la corrección


## 6. Añadir elevación usando `elevation_dict.py`

In [7]:
def busca_elevacion(row):
    return elevation_dict.get((row["Site"], row["Plot"]), np.nan)

df_all["Elevation_m"] = df_all.apply(busca_elevacion, axis=1)

faltantes = (
    df_all.loc[df_all["Elevation_m"].isna(), ["Site", "Plot"]]
    .drop_duplicates()
    .sort_values(["Site", "Plot"])
)
print(f"{df_all['Elevation_m'].notna().sum()} de {len(df_all)} filas tienen elevación asignada.\n")
print("Combinaciones (Site, Plot) SIN elevación en el diccionario (edítalo y vuelve a correr):")
print(faltantes.to_string(index=False))


399 de 672 filas tienen elevación asignada.

Combinaciones (Site, Plot) SIN elevación en el diccionario (edítalo y vuelve a correr):
      Site  Plot
   Galeras   NaN
Guacamayos   7.0
Guacamayos   8.0
Guacamayos   NaN
        SV  68.0
        SV  69.0
        SV  70.0
        SV  72.0
        SV  74.0
        SV   NaN
    Sumaco   1.0
    Sumaco   NaN
       NaN   NaN


**Nota:** las combinaciones que faltan típicamente son:
- Filas con `Plot` vacío en el archivo original (no se puede evitar).
- Parcelas de `SV` (68, 69, 70, 72, 74): el diccionario no tiene ninguna entrada para el sitio `SV` — agrégalas con el nombre de sitio real si `SV` es otro nombre.
- `Guacamayos` parcelas 7 y 8: en `elevation_dict.py` estas parcelas están registradas bajo la llave `("Galeras", 7)` / `("Galeras", 8)` en vez de `("Guacamayos", 7)` / `("Guacamayos", 8)`, por lo que no calzan con el sitio real de esas filas. Revisa si es un error de tipeo en el diccionario.
- `Sumaco` parcela 1: no existe en el diccionario (solo hay 9, 10, 11, 12, 13, 16, 18).

## 7. Árboles individuales por sitio y coincidencias entre período antiguo y reciente

Primero, cuántos árboles individuales distintos (`new_TreeID`) hay en total por sitio
(con al menos una medición, en cualquier año). Luego se compara el período
**antiguo (2005, 2006, 2007, 2011)** contra el período **reciente (2023, 2024, 2025)**,
ya que cada sitio tiene su propio año de muestreo dentro de cada bloque
(p. ej. `iWUE_OYC_GUA` va de 2005 a 2011 y se remuestreó en 2023;
`Sumaco`/`Galeras`/`SV` solo tienen 2006 y se remuestrearon en 2025).

In [8]:
arboles_por_sitio = (
    df_all.dropna(subset=["new_TreeID"])
    .groupby("Site")["new_TreeID"]
    .nunique()
    .sort_values(ascending=False)
)
print("Árboles individuales (new_TreeID únicos) con datos, por sitio:\n")
print(arboles_por_sitio.to_string())
print(f"\nTotal de árboles individuales distintos en todo el dataset: {df_all['new_TreeID'].nunique()}")


Árboles individuales (new_TreeID únicos) con datos, por sitio:

Site
SV            113
Galeras        94
Sumaco         53
Oyacachi       33
Guacamayos     28

Total de árboles individuales distintos en todo el dataset: 314


In [9]:
AÑOS_ANTIGUOS = [2005, 2006, 2007, 2011]
AÑOS_RECIENTES = [2023, 2024, 2025]

resumen = []
for site, sub in df_all.groupby("Site", dropna=False):
    ids_antiguos = set(sub.loc[sub["Year"].isin(AÑOS_ANTIGUOS), "new_TreeID"].dropna())
    ids_recientes = set(sub.loc[sub["Year"].isin(AÑOS_RECIENTES), "new_TreeID"].dropna())
    coincid = ids_antiguos & ids_recientes

    fila = {
        "Site": site,
        "arboles_unicos_totales": sub["new_TreeID"].dropna().nunique(),
        "arboles_2005_06_07_11": len(ids_antiguos),
        "arboles_2023_24_25": len(ids_recientes),
        "coincidencias_antiguo_vs_reciente": len(coincid),
    }
    # Desglose por año individual (antiguos)
    for yr in AÑOS_ANTIGUOS:
        fila[f"n_{yr}"] = sub.loc[sub["Year"] == yr, "new_TreeID"].dropna().nunique()
    # Desglose por año individual (recientes)
    for yr in AÑOS_RECIENTES:
        fila[f"n_{yr}"] = sub.loc[sub["Year"] == yr, "new_TreeID"].dropna().nunique()
    resumen.append(fila)

df_resumen = pd.DataFrame(resumen).sort_values("Site").reset_index(drop=True)
print("Árboles individuales y coincidencias entre período antiguo (2005/06/07/11) y reciente (2023/24/25):\n")
print(df_resumen.to_string(index=False))


Árboles individuales y coincidencias entre período antiguo (2005/06/07/11) y reciente (2023/24/25):

      Site  arboles_unicos_totales  arboles_2005_06_07_11  arboles_2023_24_25  coincidencias_antiguo_vs_reciente  n_2005  n_2006  n_2007  n_2011  n_2023  n_2024  n_2025
   Galeras                      94                     65                  81                                 52       0      65       0       0       0       0      81
Guacamayos                      28                     21                  28                                 21       4      20       2       0      28       0       0
  Oyacachi                      33                     26                  33                                 26       0       0       0      26      33       0       0
        SV                     113                     92                  60                                 39       0      92       0       0       0       0      60
    Sumaco                      53                    

*(La coincidencia se calcula por `new_TreeID`, el identificador más completo y
consistente entre años; `old_treeID` tiene valores faltantes o inconsistentes como `"no ID"`
en algunos archivos.)*

- `Sumaco`, `Galeras` y `SV` solo tienen mediciones en **2006** (dentro del bloque antiguo) y se remuestrearon en **2025**.
- `Oyacachi` y `Guacamayos` (dentro de `iWUE_OYC_GUA.xlsx`) tienen mediciones repartidas en **2005, 2006, 2007 y 2011**, y su remuestreo reciente es en **2023**.
- Ningún archivo tiene todavía datos de **2024**; esa columna aparecerá en 0 hasta que se agreguen.

## 8. Guardar el dataset combinado

In [10]:
SALIDA = CARPETA_DATOS / "dataset_combinado.csv"
df_all.to_csv(SALIDA, index=False)
print(f"Guardado: {SALIDA.resolve()}")
df_all.head(10)


Guardado: /home/claude/work/dataset_combinado.csv


,Site,Plot,PlotID,Year,Sample_ID,old_treeID,new_TreeID,family,genus,species,Type,C_percent,d13C_permil,N_percent,Cb_percent,d15N_permil,bulk_d13C_permil,herbarium_id,source_file,Elevation_m
0,Sumaco,9.0,Sumaco_9,2006,p09-225,225.0,NaN,Moraceae,Ficus,quijosana,None,42.578188,-28.691043,NaN,NaN,None,None,NaN,Sumaco_cellulose.xlsx,2000.0
1,Sumaco,1.0,Sumaco_1,2006,p09-230,230.0,NaN,Lauraceae,Ocotea,insularis,None,45.456318,-29.289822,NaN,NaN,None,None,3343.0,Sumaco_cellulose.xlsx,NaN
2,Sumaco,9.0,Sumaco_9,2006,p09-231,231.0,NaN,Clusiaceae,Clusia,NaN,None,45.788101,-29.783344,NaN,NaN,None,None,3342.0,Sumaco_cellulose.xlsx,2000.0
3,Sumaco,9.0,Sumaco_9,2006,p09-233,233.0,8885.0,Icacinaceae,Calatola,costaricensis,None,47.330000,-29.450000,NaN,NaN,None,None,NaN,Sumaco_cellulose.xlsx,2000.0
4,Sumaco,9.0,Sumaco_9,2006,p09-235,235.0,3733.0,Icacinaceae,Calatola,costaricensis,None,43.261474,-30.191463,NaN,NaN,None,None,NaN,Sumaco_cellulose.xlsx,2000.0
5,Sumaco,10.0,Sumaco_10,2006,p10-238,238.0,3719.0,Clusiaceae,Chrysochlamys,membranacea,None,43.175043,-29.177724,NaN,NaN,None,None,NaN,Sumaco_cellulose.xlsx,2000.0
6,Sumaco,10.0,Sumaco_10,2006,p10-239,239.0,NaN,Moraceae,Ficus,cuatrecasana,None,44.334499,-28.667583,NaN,NaN,None,None,NaN,Sumaco_cellulose.xlsx,2000.0
7,Sumaco,10.0,Sumaco_10,2006,p10-240,240.0,NaN,Lauraceae,Persea,areolatocostae,None,44.245873,-27.855772,NaN,NaN,None,None,1809.0,Sumaco_cellulose.xlsx,2000.0
8,Sumaco,10.0,Sumaco_10,2006,p10-241,241.0,8878.0,Verbenaceae,Aegiphila,NaN,None,41.620000,-28.750000,NaN,NaN,None,None,1807.0,Sumaco_cellulose.xlsx,2000.0
9,Sumaco,10.0,Sumaco_10,2006,p10-242,242.0,NaN,Solanaceae,Solanum,cf anisophyllum,None,42.914146,-29.621863,NaN,NaN,None,None,NaN,Sumaco_cellulose.xlsx,2000.0
